# Implementing a Backpropagation Package for NumPy

Build a small reverse-mode autograd engine on top of NumPy (the `Node` class and its operators), then use it on the operations graph from the previous task and on a two-layer neural network.


In [ ]:
%matplotlib inline
import sys
import utils
import numpy
import random


inputs_flat = utils.checker.load_answer('xinputs.npz')
inputs1 = [
    {'x': inputs_flat[f'inputs1 -> {i} -> x'], 'w': inputs_flat[f'inputs1 -> {i} -> w']}
    for i in range(100)
]
inputs2 = [
    {key: inputs_flat[f'inputs2 -> {i} -> {key}'] for key in ('x', 'W1', 'W2', 'b1', 'b2')}
    for i in range(100)
]

numpy.random.seed(0)
random.seed(0)

import contextlib

@contextlib.contextmanager
def safecatch():
    # Lets a later cell keep running even if an earlier, not-yet-implemented
    # method raised -- mirrors the reference course's own utils.safecatch().
    try:
        yield
    except Exception:
        import traceback
        print('!!! EXCEPTION !!!')
        print('If you see this message -- something is wrong here')
        traceback.print_exc(file=sys.stdout)
        print('!!!!!!!!!!!!!!!!!')

correct_answer = utils.checker.load_answer(
    'yanswer.npz',
    missing_ok=True,
)

answer = {}


# Implementing a Backpropagation Package for NumPy

In this project, we will implement the backpropagation algorithm for the NumPy package.
Your task will be to implement all the methods of the `Node` class, which wraps NumPy arrays and supports backpropagation.

Step-by-step, we will implement the following methods:

1. `__init__` (a constructor)
2. `backward` (a recursive mechanism that triggers backpropagation)
3. `__neg__` (negation operator $- x$)
4. `__add__` (addition operator $x + y$)
5. `__mul__` (product operator $x \cdot y$)
6. `__sub__` (subtraction operator $x - y$)
7. `__truediv__` (division operator $x / y$)
8. `exp` (exponentiation $\exp(x)$)
9. `sum` (summation $\sum_{k} x_k$)
10. `matmul` (matrix product $XY$)

After that we will use the implemented methods on:
1. a simple graph from the previous task
2. a two-layer neural network

Let's start!

`Node([1]) + Node([2]) -> Node`
`(x + y).backward()`

# Task 1

Implement the `__init__` method for the `Node` class:
- create the field `data` and assign it to the `data` argument (we will need it to compute the gradient)
- create the field `grad` and assign it to `None` (it will be used to store the gradient of the node)
- create the field `inputs` and assign it to an empty list (it will be used to store the nodes on which the current node depends)
- create the field `n_dependents` and assign it to `0` (it will be used to count the number of nodes that depend on the current node)
- create the field `n_grads` and assign it to `0` (it will be used to count the number of gradients that come to the current node)

Implement the method `update_grad` for the `Node` class. The point of this method is to take care of the gradients that come from the next nodes and accumulate them:
- first, create a deep copy of `grad_output` (just in case): `grad_output = copy.deepcopy(grad_output)`
- increase the `n_grads` counter by 1 because one of the dependent gradients has been registered
- if `grad_output` is `None` and the size of `data` is 1, assign `grad_output` to the NumPy array `[1]` (as we can calculate derivatives only of scalar values without any dependencies)
- if `grad_output` is `None`, raise the `BaseException` with the message "Backpropagation is impossible" (as we can't compute the gradient without the gradient from the next node except for the case of a scalar value)
- if `self.grad` is `None`, assign `grad_output` to `self.grad` (that means that if there was no gradient before, we will just assign the one that we got from the next node)
- otherwise, add `grad_output` to `self.grad` (that means that if there was a gradient before, we will add the one that we got from the next node to the accumulated value)

Implement the `backward` method for the `Node` class. This method is the heart of the backpropagation algorithm. Its logic is the following:
- it waits and collects gradients from all dependents of the current node
- then it calculates the list of gradients for the inputs of the current node using the `backward_step` method (we will implement this method for every operation by hand)
- trigger the `backward` methods of the inputs with the calculated gradient in the current node as an argument

As a formula, it looks like this:
$\partial_w {L} = \sum_{k=1}^{n} \partial_w {z_k} \partial_{z_k} {L}$

So, how does this work in practice?
- call the `update_grad` method with the `grad_output` argument
- if we have collected all the gradients from the dependents, it is time to calculate the gradients for the inputs of the current node 
- call the `backward_step` method with the `grad_output` argument to calculate the gradients for the inputs
- call the `backward` method of the inputs with the calculated gradient in the current node as an argument
- reset the `n_grads` counter to 0 (actually, this is not necessary as we do backpropagation only once)

The `backward_step` method should also be implemented for a basic node too. It should return an empty list because a basic node does not have any inputs.

In general, we will store inputs to the operation as a list of nodes in the `inputs` field. The `backward_step` function should return the corresponding list of gradients for the inputs.

After these steps are done, you can check that the first test passes successfully.

# Task 2

Implement the `__add__` method for the `Node` class and the class `TensorSum` that inherits from `Node`. 
This method lets us use the `+` operator for a pair of `Node` objects. After we perform `z = x + y`, the result of this operation is also a `Node` that corresponds to the sum of two nodes.

The engine for that operation is the `TensorSum` class. Here you need to:
- initialize the `Node` class with the `None` argument (empty envelope of the node) in the `__init__` method
- in the `__call__` method (this method is used when we perform `z = x + y`):
    - assign the `inputs` field of the current node to `[input1, input2]` to store the inputs
    - set `n_dependents` to the sum of `n_dependents` of the inputs (or simply to 2 because we have only two inputs)
    - calculate the value of the `data` field of the current node using the `self.inputs[0].data + self.inputs[1].data` formula
    - set the `grad` field of the current node to `None` as we don't have any gradient yet
    - return the `self` node (so that the result of the operation is the current node)
- in the `backward_step` method:
    - return the list of gradients for the inputs: `[grad_output, grad_output]` as $\partial_x {x + y} = 1$ and $\partial_y {x + y} = 1$


In `Node` class:
- implement the `__add__` method:
    - create a new `TensorSum` object with the sum of the `data` of the current node and the `other` node and return it (`return TensorSum()(self, other)`)

Now your `Node` class supports the `+` operator. You can check it by running test number 2. So you can calculate gradients of expressions like `x + y + x + x + y + y`. That is not much, but it is a good start.

# Task 3 and others

Implement the remaining operations by analogy:
- `__sub__` that will be used when we perform `z = x - y`
- `__mul__` that will be used when we perform `z = x * y`
- `__truediv__` that will be used when we perform `z = x / y`
- `__neg__` that will be used when we perform `z = -x`
- `exp` that will be used when we perform `z = numpy.exp(x)`
- `sum` that will be used when we perform `z = numpy.sum(x)`
- `matmul` that will be used when we perform `z = x @ y`

In all these cases, use theoretical derivatives to implement the `backward_step` function for each of these operations.

The hardest one will be `matmul` as it is a matrix multiplication. To implement it, use the following rule of thumb:
- the resulting gradient should have the same shape as the data
- the gradients of linear functions are also linear functions

In [ ]:
# Edit this cell in task.ipynb.
import numpy
import copy

# Node(np.array([1]))
class Node:
    # Defaults, so printing or reading a not-yet-initialised node does not crash.
    data = None
    grad = None

    def __init__(self, data):
        ...
        ### YOUR CODE HERE ###

    def update_grad(self, grad_output):
        ...
        ### YOUR CODE HERE ###

    def backward(self, grad_output=None):
        ...
        ### YOUR CODE HERE ###

    def backward_step(self, grad_output=None):
        ...
        ### YOUR CODE HERE ###

    # @backprop
    # def backward(self, grad_output=None):


    def __str__(self):
        return f"Node(data={self.data}, grad={self.grad})"

    def __neg__(self):
        return TensorNeg()(self)

    def __add__(self, other):
        return TensorSum()(self, other)

    def __mul__(self, other):
        return TensorMul()(self, other)

    def __sub__(self, other):
        return TensorSub()(self, other)

    def __truediv__(self, other):
        return TensorDiv()(self, other)

    def exp(self):
        return TensorExp()(self)

    def sum(self, axis=None):
        return TensorSumReduce()(self)

    def __matmul__(self, other):
        return TensorMatMul()(self, other)


class TensorSum(Node):
    def __init__(self):
        ...
        ### YOUR CODE HERE ###

    def __call__(self, input1, input2):
        ...
        ### YOUR CODE HERE ###
        return self

    def backward_step(self, grad_output=None):
        ...
        ### YOUR CODE HERE ###


class TensorSub(Node):
    def __init__(self):
        ...
        ### YOUR CODE HERE ###

    def __call__(self, input1, input2):
        ...
        ### YOUR CODE HERE ###
        return self

    def backward_step(self, grad_output=None):
        ...
        ### YOUR CODE HERE ###


class TensorMul(Node):
    def __init__(self):
        ...
        ### YOUR CODE HERE ###

    def __call__(self, input1, input2):
        ...
        ### YOUR CODE HERE ###
        return self

    def backward_step(self, grad_output=None):
        ...
        ### YOUR CODE HERE ###


class TensorDiv(Node):
    def __init__(self):
        ...
        ### YOUR CODE HERE ###

    def __call__(self, input1, input2):
        ...
        ### YOUR CODE HERE ###
        return self

    def backward_step(self, grad_output=None):
        ...
        ### YOUR CODE HERE ###


class TensorNeg(Node):
    def __init__(self):
        ...
        ### YOUR CODE HERE ###

    def __call__(self, input1):
        ...
        ### YOUR CODE HERE ###
        return self

    def backward_step(self, grad_output=None):
        ...
        ### YOUR CODE HERE ###


class TensorExp(Node):
    def __init__(self):
        ...
        ### YOUR CODE HERE ###

    def __call__(self, input1):
        ...
        ### YOUR CODE HERE ###
        return self

    def backward_step(self, grad_output=None):
        ...
        ### YOUR CODE HERE ###


class TensorSumReduce(Node):
    def __init__(self):
        ...
        ### YOUR CODE HERE ###

    def __call__(self, input1):
        ...
        ### YOUR CODE HERE ###
        return self

    def backward_step(self, grad_output=None):
        ...
        ### YOUR CODE HERE ###


class TensorMatMul(Node):
    def __init__(self):
        ...
        ### YOUR CODE HERE ###

    def __call__(self, input1, input2):
        ...
        ### YOUR CODE HERE ###
        return self

    def backward_step(self, grad_output=None):
        ...
        ### YOUR CODE HERE ###


In [ ]:
# TEST 1

with safecatch():
    x = Node(numpy.array([1]))
    print(x)
    x.backward()
    print(x)

    answer['init'] = []
    for inp in inputs1:
        x = Node(inp['x'][0])

        x.backward()

        answer['init'].append(x.grad)


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'init'))

In [ ]:
# TEST 2

with safecatch():
    x = Node(numpy.array([1]))
    y = Node(numpy.array([2]))

    z = x + y + x + x + x + y

    print(z)
    z.backward()
    print(x, y)

    answer['sum'] = []
    for inp in inputs1:
        x = Node(inp['x'][0])
        y = Node(inp['x'][1])

        z = x + y + x + x + x + y
        z.backward()

        answer['sum'].append(x.grad)


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'sum'))

In [ ]:
# TEST 3

with safecatch():
    x = Node(numpy.array([1]))
    y = Node(numpy.array([2]))

    z = x - y + x + x - x - y

    print(z)
    z.backward()
    print(x, y)

    answer['diff'] = []
    for inp in inputs1:
        x = Node(inp['x'][0])
        y = Node(inp['x'][1])

        z = x - y + x + x - y - y
        z.backward()

        answer['diff'].append(x.grad)


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'diff'))

In [ ]:
# TEST 4

with safecatch():
    x = Node(numpy.array([1]))
    y = Node(numpy.array([2]))

    z = (x + y) * (x - y)

    print(z)
    z.backward()
    print(x, y)

    answer['mul'] = []
    for inp in inputs1:
        x = Node(inp['x'][0])
        y = Node(inp['x'][1])

        z = (x + y) * (x - y) * (x + x + y)
        z.backward()
        answer['mul'].append(x.grad)


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'mul'))

In [ ]:
print()
print(answer['mul'])


In [ ]:
# TEST 5

with safecatch():
    x = Node(numpy.array([1]))
    y = Node(numpy.array([2]))

    z = x / y

    print(z)
    z.backward()
    print(x, y)

    answer['div'] = []
    for inp in inputs1:
        x = Node(inp['x'][0])
        y = Node(inp['x'][1])

        z = x / (Node(0.5) + y)
        z.backward()

        answer['div'].append(x.grad)


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'div'))

In [ ]:
# TEST 6
with safecatch():
    x = Node(numpy.array([1]))
    y = Node(numpy.array([2]))

    z = -x

    print(z)
    z.backward()
    print(x)

    answer['neg'] = []
    for inp in inputs1:
        x = Node(inp['x'][0])

        z = -x
        z.backward()

        answer['neg'].append(x.grad)


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'neg'))

In [ ]:
# TEST 7

with safecatch():
    x = Node(numpy.array([1]))
    y = Node(numpy.array([2]))

    z = (x + y).exp()

    print(z)
    print(x.grad, y.grad)

    z.backward()

    print(x.grad, y.grad)

    answer['exp'] = []
    for inp in inputs1:
        x = Node(inp['x'][0])

        z = x.exp()
        z.backward()

        answer['exp'].append(x.grad)


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'exp'))

# Task

Implement the graph function from the previous task. For constants, please use `Node(1)` whenever needed (otherwise you will get an error)

In [ ]:
# Edit this cell in task.ipynb.
# TEST 8 (Graph)

def sigmoid(z):
    ...
    ### YOUR CODE HERE ###

def tanh(z):
    ...
    ### YOUR CODE HERE ###

def graph_value(x, w):
    y = x
    ## YOUR CODE HERE ##
    return y

with safecatch():
    answer['graph'] = []
    for inp in inputs1:

        x = inp['x']
        w = inp['w']

        x = [Node(float(val)) for val in x]
        w = [Node(float(val)) for val in w]

        y = graph_value(x, w)
        y.backward()

        answer['graph'].append([x[0].grad, x[1].grad, w[0].grad, w[1].grad, w[2].grad, w[3].grad])

    print("OUR RESULTS:")
    print(x[0].grad, x[1].grad, w[0].grad, w[1].grad, w[2].grad, w[3].grad)


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'graph'))

# 2-layer NN

Implement a two-layer neural network and compute its gradient using the `Node` class:

$\mathbf y = \sigma( W_2 \sigma(W_1 \mathbf x + \mathbf b_1) + \mathbf b_2)$

Return the sum of all values in $y * y$ as the loss function.

In [ ]:
# Edit this cell in task.ipynb.
# TEST 9 (Two-layer net)

def two_layer_net(x, W1, W2, b1, b2):
    ...
    ## YOUR CODE HERE ##
    return y.sum()


with safecatch():
    answer['two_layer_net'] = []
    for inp in inputs2:
        x = Node(inp['x'])
        W1 = Node(inp['W1'])
        W2 = Node(inp['W2'])
        b1 = Node(inp['b1'])
        b2 = Node(inp['b2'])

        h_hat = two_layer_net(x, W1, W2, b1, b2)
        h_hat.backward()

        answer['two_layer_net'].append([x.grad, W1.grad, W2.grad, b1.grad, b2.grad])


In [ ]:
utils.checker.print_report(utils.checker.run_checks(answer, correct_answer, 'two_layer_net'))

# Conclusion

You have implemented a backpropagation algorithm. This algorithm is similar to the one used in PyTorch. Note that you have implemented all the mechanics of it. Thus, it should no longer be a magic box: you know exactly how it works.

In [ ]:
def replace_none_with_nan(value):
    # A gradient that was never computed is None; the answer file cannot store it.
    if value is None:
        return numpy.nan
    if isinstance(value, dict):
        return {key: replace_none_with_nan(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [replace_none_with_nan(item) for item in value]
    return value


utils.checker.save_npz(replace_none_with_nan(answer), str(utils.checker.get_data_dir() / 'answer.npz'))
